# Project title
## MSE course EVA_CE4IB 2026

Authors: Drigatti Alessio, Zurbrügg Milena

[![Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/drigaa/rEVA_CE4IB/HEAD?urlpath=%2Fdoc%2Ftree%2Fcasestudy.ipynb)

## Project Overview
The objective of this project is to simulate the thermal dynamics of two rooms located at the corner of a building. Figure 1 illustrates the floor plan of the area under consideration including the relevant measurements. Each room has a height of 3 m.

![thermal-network](images/Room_Plan.png)

Figure 1: Floor Plan of the considered area.

The two grey areas represent the rooms considered in this project. Room 5, located on the right-hand side, is situated at the corner of the building and therefore has two external façades. One external side consists of an insulated concrete wall (orange), while the other contains a window (blue). Room 5 is connected to Room 9 through a non-insulated concrete wall (orange).

Room 9 is adjacent to the exterior through a single window (blue). The regions highlighted in red are assumed to have no influence on the thermal behavior of the rooms, as they are considered to be exposed to the same boundary conditions as the indoor environment and are therefore neglected in the analysis. The same is assumed for the floor an ceiling of the rooms.




In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from tqdm import tqdm

import dm4bem as db

In [ ]:
from settings_simulation import *

## Thermal Network

The thermal behaviour of the two rooms is represented by a lumped-parameter thermal network based on thermal resistances, thermal capacitances, temperature sources, and heat flow sources. Figure 2 illustrates the complete network of the considered area.

![thermal-network](images/Screenshot%202026-10-29%20093356.png)

Figure 2: Thermal circuit for the considered area shown in Figure 1.

The outdoor temperature *T<sub>0</sub>* acts as the external boundary condition for all building envelope components. Room air temperatures are represented by nodes 5 and 9, corresponding to Rooms 5 and 9 in the floor plan, respectively. Each room is connected to a temperature source *T<sub>i,sp</sub>*, representing the HVAC control system that maintains the indoor air temperature at the prescribed setpoint. The conductances in branches 10 and 11 model the heating or cooling power that the HVAC control system uses to keep the rooms at their setpoint temperatures.

### Exterior Corner Wall of Room 5

The insulated exterior wall of Room 5 is shown in Figure 2 in orange, going from node 0 to node 4. Heat is transferred from the outdoor environment through an external convective resistance (branch 0) into the wall structure. Solar radiation absorbed by the outer wall surface is represented by the heat flow source *Φ<sub>0</sub>*, which is connected to node 0.

The wall itself consists of concrete and insulation layers discretized into four conductive branches (1-4). Thermal masses are assigned to nodes 1 and 3, representing the heat storage capacity of the wall materials. Heat is finally transferred to the room air through an internal convective resistance (branch 5), connecting the wall to the indoor air node of Room 5 (node 5).

### Internal Wall Between Rooms

The separating wall between Rooms 5 and 9 consists of non-insulated concrete and is represented by nodes 6 to 8. Convective heat transfer occurs on both wall surfaces (branches 6 and 9), while conductive heat transport through the concrete is described by branches 7 and 8. A thermal capacitance is assigned to node 7 to account for heat storage within the wall.

This wall establishes thermal coupling between the two rooms, allowing heat transfer whenever a temperature difference exists between nodes 5 and 9.

### Room Air Nodes

The indoor air of each room is represented by a thermal capacitance connected to nodes 5 and 9. These capacitances describe the thermal storage capacity of the room air and the interior contents lumped into a single node.

Both rooms are subjected to internal heat gains *Q<sub>aux</sub>*, representing occupants, lighting, electrical devices, and other internal heat sources. These gains are directly added to the respective room air nodes.

Additionally, solar radiation transmitted through the windows and absorbed within the rooms is represented by the heat flow source *Φ<sub>i</sub>*.

### Windows

Each room contains a window exposed to the outdoor environment. Heat transfer through the windows is represented by branches 12 and 13, connecting the outdoor temperature source *T<sub>0</sub>* directly to the room air nodes through an equivalent thermal conductance.

**!!!** **IMPORTANT** **!!!**

Change the simulation settings [here](settings_simulation.py) and re-run notebook.

In [ ]:
number_of_nodes = 10
number_of_branches = 14

air = {
    'Density': 1.2,
    'Specific heat': 1000
}

# convection coefficients, W/(m²·K)
h = pd.DataFrame([{'in': 4., 'out': 10.}], index=['h'])

l = 3

concrete = {
    'Conductivity': 1.400,
    'Density': 2300.0,
    'Specific heat': 880,
    'Width': 0.15,
    'Surface': l ** 2
}

insulation = {
    'Conductivity': 0.040,
    'Density': 16.0,
    'Specific heat': 1210,
    'Width': 0.25,
    'Surface': l ** 2
}

glass = {
    'U-Value': 0.6,
    'Surface': l ** 2
}

wall = pd.DataFrame.from_dict(
    {
        'Layer_out': insulation,
        'Layer_in': concrete,
        'window': glass
    },
    orient='index')

wall

## System of algebraic-differential equations DAE
### Temp nodes and flow branches

The thermal network is described by the branch heat-flow equation

\begin{equation}
q =  G(-A*θ + b),
\end{equation}

where q is the vector of heat flows through the network branches,

\begin{equation}
q =
\begin{bmatrix}
q_0 & q_1 & q_2 & q_3 & q_4 & q_5 &
q_6 & q_7 & q_8 & q_9 & q_{10} & q_{11} & q_{12} & q_{13}
\end{bmatrix}
^{T}.
\end{equation}

The temperature state vector

\begin{equation}
θ =
\begin{bmatrix}
\theta_0 & \theta_1 & \theta_2 & \theta_3 &
\theta_4 & \theta_5 & \theta_6 & \theta_7 & \theta_8 & \theta_9
\end{bmatrix}^{T}
\end{equation}

contains the temperatures of the eight thermal nodes.

In [ ]:
θ = [f'θ{i}' for i in range(number_of_nodes)]
q = [f'q{i}' for i in range(number_of_branches)]

#### Incidence matrix $A$
The incidence matrix defines the connectivity between the 14 heat-flow branches and the 10 temperature nodes. A value of $1$ indicates if the flow goes into the node, while $-1$ indicates if it gets out of the node.

In [ ]:
A = np.zeros((number_of_branches, number_of_nodes))

A[0, 0] = 1
A[1, 0], A[1, 1] = -1, 1
A[2, 1], A[2, 2] = -1, 1
A[3, 2], A[3, 3] = -1, 1
A[4, 3], A[4, 4] = -1, 1
A[5, 4], A[5, 5] = -1, 1
A[6, 6], A[6, 5] = -1, 1
A[7, 7], A[7, 6] = -1, 1
A[8, 8], A[8, 7] = -1, 1
A[9, 9], A[9, 8] = -1, 1
A[10, 9] = 1
A[11, 5] = 1
A[12, 9] = 1
A[13, 5] = 1

A = pd.DataFrame(A, index=q, columns=θ)

### Conductance Matrix $G$

The diagonal conductance matrix contains the thermal conductances of all branches.

Conductances 12 and 13 represent the heat transfer through the windows and are determined from the corresponding  [U-values](https://www.neuffer.de/en/u-values.php). It describes the rate at which heat flows through a material or building component from the warmer side to the cooler side. A smaller U-value indicates better thermal insulation and lower heat loss. Modern high-performance glazing can achieve very low U-values. For instance, triple-pane windows commonly reach values ranging from approximately 0.4 to 0.8 W/m²K, with around 0.6 W/m²K being considered an excellent level of thermal performance by current standards.



In [ ]:
## P-controler gain
## Will be overwritten later in the code
Kp = 0

G = np.zeros(number_of_branches)

G[[1, 2]] = (wall["Conductivity"] / (wall["Width"] / 2) * wall["Surface"]).Layer_out
G[[3, 4, 7, 8]] = (wall["Conductivity"] / (wall["Width"] / 2) * wall["Surface"]).Layer_in

G[[0]] = h['out'] * wall["Surface"].Layer_out
G[[5, 6, 9]] = h['in'] * wall["Surface"].Layer_in

G[[12, 13]] = (wall['U-Value'] * wall["Surface"]).window

G[[10, 11]] = Kp
G = pd.Series(G, index=q)
G

### Capacity Matrix $C$

The dynamic behaviour of the system is represented by the capacitance matrix

\begin{equation}
\mathbf C =
\operatorname{diag}
(0, C_1, 0, C_3, 0, C_5, 0, C_7, 0, C_9),
\end{equation}

which contains the thermal capacities associated with the storage nodes of the network. Nodes with zero entries are assumed to have negligible thermal storage.

In [ ]:
C = np.zeros(number_of_nodes)

C[1] = (wall['Width'] * wall['Surface'] * wall['Density'] * wall["Specific heat"]).Layer_out
C[[3, 7]] = (wall['Width'] * wall['Surface'] * wall['Density'] * wall["Specific heat"]).Layer_in

if air_with_capacity:
    C[[5, 9]] = air["Density"] * l ** 3 * air["Specific heat"]

C = pd.Series(C, index=θ)
C

### Temperatur source vector $b$

The boundary-condition vector

\begin{equation}
b =
\begin{bmatrix}
T_o & 0 & 0 & 0 & 0 & 0 & 0 & 0 &
0 & 0 & T_{isp} & T_{isp} & T_o & T_o
\end{bmatrix}^{T}
\end{equation}

introduces prescribed temperatures into the network, where $T_o$ denotes the outdoor temperature and $T_{isp}$ the indoor temperature setpoint.

In [ ]:
b = pd.Series(np.zeros(number_of_branches), index=q)
b[[0, 12, 13]] = "T_0"
b[[10, 11]] = "Ti_sp"
b

    ### Heat flow source vector $f$

Finally, the source vector

\begin{equation}
f =
\begin{bmatrix}
\Phi_o &
0 &
0 &
0 &
\Phi_i &
\dot Q_{aux} &
\Phi_i &
0 &
\Phi_i &
\dot Q_{aux} &
\end{bmatrix}^{T}
\end{equation}

contains all external heat inputs. Here, $\Phi_o$ represents solar and environmental heat gains, $\Phi_i$ the internal heat gains, $\dot Q_a$ the ventilation or air-flow heat exchange rate, and $\Phi_a$ the heat supplied by the HVAC system. ???

In [ ]:
f = pd.Series(np.zeros(number_of_nodes), index=θ)
f[[0]] = 'Φo'
f[[5, 9]] = 'Qa'
f[[4, 6, 8]] = 'Φi'
f

### Output vector $y$

In [ ]:
y = pd.Series(np.zeros(number_of_nodes), index=θ)
y[[5, 9]] = 1
y

In [ ]:
TC = {"A": A,
      "G": G,
      "C": C,
      "b": b,
      "f": f,
      "y": y}

## Steady state analysis from DAE

In [ ]:
if controller:
    TC["G"][["q10", "q11"]] = Kp_value

In [ ]:
# State-space
[As, Bs, Cs, Ds, us] = db.tc2ss(TC)

##### Overwrite $b$ and $f$ vector for free floating building (no controller)

In [ ]:
b_ss = np.zeros(number_of_branches)
b_ss[[0, 12, 13]] = To_ss
b_ss[[10, 11]] = Ti_sp_ss

f_ss = np.zeros(number_of_nodes)

A = TC['A']
G = TC['G']
diag_G = pd.DataFrame(np.diag(G), index=G.index, columns=G.index)

θss = np.linalg.inv(A.T @ diag_G @ A) @ (A.T @ diag_G @ b_ss + f_ss)
print("Results free floating building")
print(f'θss = {np.around(θss, 2)} °C')

In [ ]:
b_ss = np.zeros(number_of_branches)
b_ss[[0, 12, 13]] = To_ss
b_ss[[10, 11]] = Ti_sp_ss

f_ss = np.zeros(number_of_nodes)
f_ss[[5, 9]] = Qa_ss

A = TC['A']
G = TC['G']
diag_G = pd.DataFrame(np.diag(G), index=G.index, columns=G.index)

θss_Q = np.linalg.inv(A.T @ diag_G @ A) @ (A.T @ diag_G @ b_ss + f_ss)
print(f"Results with auxiliary heat source {Qa_ss} Watts on node (room) 5 and 9")
print(f'θss_Q = {np.around(θss_Q, 2)} °C')

## Steady State analysis from state-space represintation

In [ ]:
us

In [ ]:
## Create steady state input vector

bT = np.array([To_ss, Ti_sp_ss, Ti_sp_ss, To_ss, To_ss])
fQ = np.array([0, 0, Qa_ss, 0, 0, Qa_ss])
uss = np.hstack([bT, fQ])
uss = pd.Series(uss, index=us.index)
uss

In [ ]:
inv_As = pd.DataFrame(np.linalg.inv(As),
                      columns=As.index, index=As.index)
yss = (-Cs @ inv_As @ Bs + Ds) @ uss

print(f"Node/Room {yss.index[0]}: {yss[0]:.2f}")
print(f"Node/Room {yss.index[1]}: {yss[1]:.2f}")

In [ ]:
print(f"Error between DAE and state-space\n{abs(yss - θss_Q[[5, 9]])}")

## Eigenvalue analysis
### Time step

In [ ]:
λ = np.linalg.eig(As)[0]

# time step
Δtmax = 2 * min(-1. / λ)  # max time step for stability of Euler explicit
db.print_rounded_time('Δtmax', Δtmax)

if imposed_time_step:
    dt = Δt
else:
    dt = db.round_time(Δtmax)

db.print_rounded_time('dt', dt)
print(f"dt = {dt:.0f} s")

### Settling time

In [ ]:
# settling time
t_settle = 4 * max(-1 / λ)
db.print_rounded_time('t_settle', t_settle)

# duration: next multiple of 3600 s that is larger than t_settle
duration = np.ceil(t_settle / 3600) * 3600
db.print_rounded_time('duration', duration)

## Step response to outdoor temperature
### Input vector

In [ ]:
filename = Path("weather_data/Hombrechtikon/CHE_ZH_Hombrechtikom.068510_TMYx.2011-2025.epw")
[data, meta] = db.read_epw(filename, coerce_year=None)
weather = data[["temp_air", "dir_n_rad", "dif_h_rad"]]

In [ ]:
weather.index = weather.index.map(lambda t: t.replace(year=2000))
weather = weather.loc[start_date:end_date]

In [ ]:
surface_east = {'slope': slope,
                'azimuth': azimuth_east,
                'latitude': 45}

surface_north = {'slope': slope,
                'azimuth': azimuth_north,
                'latitude': 45}

Etot_east = db.sol_rad_tilt_surf(
    weather, surface_east, albedo).sum(axis=1)

Etot_north = db.sol_rad_tilt_surf(
    weather, surface_north, albedo).sum(axis=1)

In [ ]:
# Create input_data_set
n = weather.shape[0]

To = weather["temp_air"]  # outdoor temperature
Ti_sp = temp_indor_setpoint * np.ones(n)  # indoor temperature set point
Φo = wall_area * alpha * Etot_east
Φi = wall_area * view_factor * beta_window * Etot_north
Qa = Qa * np.ones(n)

data = {'T_0': To, 'Ti_sp': Ti_sp, 'Φo': Φo, 'Φi': Φi, 'Qa': Qa}
input_data_set = pd.DataFrame(data, index=weather.index).resample(str(dt) + 'S').interpolate(method="linear")

u = db.inputs_in_time(us, input_data_set)

### Time integration

In [ ]:
# Initial conditions
θ_exp = pd.DataFrame(index=u.index)  # empty df with index for explicit Euler
θ_imp = pd.DataFrame(index=u.index)  # empty df with index for implicit Euler

θ_exp[As.columns] = temp_init  # fill θ for Euler explicit with initial values θ0
θ_imp[As.columns] = temp_init  # fill θ for Euler implicit with initial values θ0

I = np.eye(As.shape[0])

for k in tqdm(range(u.shape[0] - 1)):
    θ_exp.iloc[k + 1] = (I + dt * As) @ θ_exp.iloc[k] + dt * Bs @ u.iloc[k]
    θ_imp.iloc[k + 1] = np.linalg.inv(I - dt * As) @ (θ_imp.iloc[k] + dt * Bs @ u.iloc[k])

# outputs
y_exp = (Cs @ θ_exp.T + Ds @ u.T).T
y_imp = (Cs @ θ_imp.T + Ds @ u.T).T

In [ ]:
# plot results
y = pd.DataFrame({
    'Explicit': y_exp.iloc[:, 0].to_numpy(dtype=float),
    'Implicit': y_imp.iloc[:, 0].to_numpy(dtype=float),
}, index=y_exp.index)

# keep at most ~5000 points
n = max(1, len(y) // 3000)
y_plot = y.iloc[::n]

ax = y_plot.plot()
ax.set_xlabel('Time')
ax.set_ylabel('Indoor temperature, $\\theta_i$ / °C')
ax.set_title(f'Node/Room 5\nTime step: $dt$ = {dt:.0f} s; $dt_{{max}}$ = {Δtmax:.0f} s')
ax.set_ylim(19, 26)   # re-enable once you see the actual range
plt.show()

In [ ]:
# plot results
y = pd.DataFrame({
    'Explicit': y_exp.iloc[:, 1].to_numpy(dtype=float),
    'Implicit': y_imp.iloc[:, 1].to_numpy(dtype=float),
}, index=y_exp.index)

# keep at most ~5000 points
n = max(1, len(y) // 3000)
y_plot = y.iloc[::n]

ax = y_plot.plot()
ax.set_xlabel('Time')
ax.set_ylabel('Indoor temperature, $\\theta_i$ / °C')
ax.set_title(f'Node/Room 9\nTime step: $dt$ = {dt:.0f} s; $dt_{{max}}$ = {Δtmax:.0f} s')
ax.set_ylim(19, 24)   # re-enable once you see the actual range
plt.show()

In [ ]:
Kp5 = TC["G"]["q11"]
q_HVAC_5 = Kp5 * (u["q11"] - y_imp["θ5"]) / l ** 2
ax = q_HVAC_5.plot()
ax.set_ylabel(r"Heating or cooling Power $q$ / $(W \cdot m^{-2})$")
ax.set_xlabel(r"Time")
ax.set_title("Heating or Cooling over the year in room 5")
plt.show()